In [ ]:
# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements.  See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership.  The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License.  You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied.  See the License for the
# specific language governing permissions and limitations
# under the License.

In [ ]:
!pip install apache-burr[start]

# Artifact / object storage

This notebook shows how to use Burr's artifact (blob) store to keep large values out of
`State` directly. An action writes bytes to a store (here, the local filesystem, so this
notebook needs no cloud credentials) and keeps only a small, content-addressed `ArtifactRef`
in state; a later action reads the bytes back out using that reference.

In [ ]:
import tempfile

from burr.core import ApplicationBuilder, ApplicationContext, Result, State, action
from burr.core.artifacts import ArtifactRef, LocalFileSystemArtifactStore

## Define actions

`write_document` writes `content` to the object store and keeps only the resulting
`ArtifactRef` in state. `read_document` reads the artifact back out using that reference.

In [ ]:
@action(reads=[], writes=["document"])
def write_document(state: State, __context: ApplicationContext, content: bytes) -> State:
    ref: ArtifactRef = __context.object_store.put_artifact(content, media_type="text/plain")
    print(f"Wrote {ref.size_bytes} bytes to the object store under key '{ref.key}'")
    return state.update(document=ref)


@action(reads=["document"], writes=["recovered_text"])
def read_document(state: State, __context: ApplicationContext) -> State:
    recovered_bytes = __context.object_store.get_artifact(state["document"])
    return state.update(recovered_text=recovered_bytes.decode("utf-8"))

## Build the application

`with_object_store(...)` makes the store available to every action through
`ApplicationContext.object_store`.

In [ ]:
object_store = LocalFileSystemArtifactStore(tempfile.mkdtemp())

app = (
    ApplicationBuilder()
    .with_actions(
        write_document=write_document,
        read_document=read_document,
        result=Result("recovered_text"),
    )
    .with_transitions(
        ("write_document", "read_document"),
        ("read_document", "result"),
    )
    .with_entrypoint("write_document")
    .with_state()
    .with_object_store(object_store)
    .build()
)
app

## Run it

In [ ]:
original_content = b"This document is too large to keep in state directly."
action_, result, state = app.run(halt_after=["result"], inputs={"content": original_content})
print(f"Recovered: {state['recovered_text']!r}")
assert state["recovered_text"] == original_content.decode("utf-8")

## Async applications and S3

This notebook sticks to the synchronous, local-filesystem store to avoid requiring AWS
credentials. If you're building an async application (one built with `abuild()`), use
`AsyncS3ArtifactStore` instead -- it's backed by `aiobotocore` so it won't block the event
loop. It owns an underlying client that should be cleaned up with `async with`:

```python
async with await AsyncS3ArtifactStore.acreate(bucket="my-bucket") as object_store:
    app = await ApplicationBuilder()...with_object_store(object_store).abuild()
    await app.arun(...)
# the underlying aiobotocore client is closed automatically here, even on exception.
```

See the [artifact storage concept docs](https://burr.apache.org/concepts/artifact-storage/) for
more detail.